# Tâche 3 : Visualisation des effets de site dans les données fMRI

## Contexte

Les tâches 1 et 2 ont montré que les performances du modèle varient considérablement selon les sites d'acquisition, et que cette variabilité n'est pas uniquement expliquée par l'âge des participants. La tâche 3 remonte en amont de la chaîne d'analyse pour examiner si ces effets de site sont **visibles directement dans les données fMRI brutes**, avant même toute extraction de features.

## Objectif

Produire deux types de visualisations par site :

1. **Volume moyen par site** : moyenne voxel par voxel de toutes les images fMRI d'un site. Les différences d'intensité, de champ de vue ou de contraste entre sites seront directement visibles.
2. **Carte d'écart-type temporel (tSD) par site** : pour chaque sujet, calculer l'écart-type voxel par voxel sur la série temporelle (axe temps), puis moyenner par site. Les zones avec une forte variabilité temporelle révèlent des artefacts dominants.

## Lien avec les tâches précédentes

- **Tâche 1** : les performances LOSO varient entre 0.44 (OHSU) et 0.75 (LEUVEN) selon le site
- **Tâche 2** : cette variabilité n'est pas uniquement due à l'âge
- **Tâche 3** : on cherche à visualiser si les données fMRI brutes présentent des différences inter-sites qui pourraient expliquer ces résultats

## Instructions de reproduction

Ce notebook nécessite les fichiers `func_preproc.nii.gz` téléchargés via `prepare_data.py`.

```bash
git clone https://github.com/evavilleneuve/abide-fmri.git
cd abide-fmri
python3 -m venv venv_abide
source venv_abide/bin/activate
pip install -r requirements-modern.txt
mkdir -p data output
python code/prepare_data.py data output
```

Les fichiers fMRI sont attendus dans `data/ABIDE_pcp/cpac/nofilt_noglobal/`.

## 1. Imports

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib

from nilearn import datasets, plotting, image

In [2]:
# Style uniforme pour toutes les figures
plt.rcParams['axes.formatter.useoffset'] = False
plt.rcParams['axes.formatter.limits'] = (-8000000, 8000000)
plt.rcParams['figure.dpi'] = 150

## 2. Chargement du phénotype et mapping des sites

In [ ]:
project_root = Path.cwd().parent
fmri_dir     = project_root / "data" / "ABIDE_pcp" / "cpac" / "nofilt_noglobal"
data_dir     = project_root / "data"

assert fmri_dir.exists(), f"Dossier fMRI introuvable : {fmri_dir}"

# Charger le phénotype
abide = datasets.fetch_abide_pcp(
    data_dir=str(data_dir),
    pipeline="cpac",
    quality_checked=True
)
pheno = pd.DataFrame(abide.phenotypic)
pheno["DX"] = pheno["DX_GROUP"].map({1: "ASD", 2: "TD"})

print(f"Participants dans le phénotype : {len(pheno)}")
print(f"Sites : {sorted(pheno['SITE_ID'].unique())}")

[fetch_abide_pcp] Dataset found in /mnt/d/udem/PSY3019/Francois_Presentation_Projet_ABIDE-fMRI/data/ABIDE_pcp
[fetch_abide_pcp] Downloading data from https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/nofilt_noglobal/func_preproc/Pitt_0050003_func_preproc.nii.gz ...
Downloaded 47816704 of 104419884 bytes (45.8%%,  9.7min remaining)
Downloaded 47955968 of 104419884 bytes (45.9%%,  8.4min remaining)
Downloaded 48046080 of 104419884 bytes (46.0%%,  9.4min remaining)
Downloaded 48095232 of 104419884 bytes (46.1%%, 11.7min remaining)
Downloaded 48168960 of 104419884 bytes (46.1%%, 12.3min remaining)
Downloaded 48218112 of 104419884 bytes (46.2%%, 13.6min remaining)
Downloaded 48267264 of 104419884 bytes (46.2%%, 14.6min remaining)
Downloaded 48324608 of 104419884 bytes (46.3%%, 15.0min remaining)
Downloaded 48406528 of 104419884 bytes (46.4%%, 14.7min remaining)
Downloaded 48496640 of 104419884 bytes (46.4%%, 14.4min remaining)
Downloaded 48562176 of 104419884 byt

In [ ]:
# Lister tous les fichiers fMRI disponibles
fmri_files = sorted(fmri_dir.glob("*_func_preproc.nii.gz"))
print(f"Fichiers fMRI trouvés : {len(fmri_files)}")

# Construire un DataFrame fichier → site
# Le nom de fichier est de la forme : SiteName_SubjectID_func_preproc.nii.gz
# Ex : Pitt_0050003_func_preproc.nii.gz → site = Pitt
#      CMU_a_0050642_func_preproc.nii.gz → site = CMU_a
file_records = []
for f in fmri_files:
    # Nom sans extension : ex. Leuven_1_0050682_func_preproc
    stem = f.name.replace("_func_preproc.nii.gz", "")
    parts = stem.split("_")
    
    # Le subject ID est toujours un nombre à 7 chiffres (005XXXX)
    sub_id = None
    sub_idx = None
    for i, p in enumerate(parts):
        if len(p) == 7 and p.isdigit():
            sub_id = int(p)
            sub_idx = i
            break
    
    if sub_id is not None:
        site_name = "_".join(parts[:sub_idx])
        file_records.append({
            "filepath":  f,
            "site_file": site_name,
            "sub_id":    sub_id
        })
    else:
        warnings.warn(f"Impossible de parser : {f.name}", UserWarning)

df_files = pd.DataFrame(file_records)
print(f"Fichiers parsés : {len(df_files)}")
print(f"Sites trouvés : {sorted(df_files['site_file'].unique())}")

In [ ]:
# Mapping entre noms de sites dans les fichiers et noms dans le phénotype
# Les noms de fichiers utilisent des conventions différentes du phénotype
site_mapping = {
    "Caltech":  "CALTECH",
    "CMU_a":    "CMU",
    "CMU_b":    "CMU",
    "KKI":      "KKI",
    "Leuven_1": "LEUVEN_1",
    "Leuven_2": "LEUVEN_2",
    "MaxMun_a": "MAX_MUN",
    "MaxMun_b": "MAX_MUN",
    "MaxMun_c": "MAX_MUN",
    "MaxMun_d": "MAX_MUN",
    "NYU":      "NYU",
    "OHSU":     "OHSU",
    "Olin":     "OLIN",
    "Pitt":     "PITT",
    "SBL":      "SBL",
    "SDSU":     "SDSU",
    "Stanford": "STANFORD",
    "Trinity":  "TRINITY",
    "UCLA_1":   "UCLA_1",
    "UCLA_2":   "UCLA_2",
    "UM_1":     "UM_1",
    "UM_2":     "UM_2",
    "USM":      "USM",
    "Yale":     "YALE",
}

df_files["site"] = df_files["site_file"].map(site_mapping)

# Vérifier les sites non mappés
unmapped = df_files[df_files["site"].isna()]["site_file"].unique()
if len(unmapped) > 0:
    print(f" Sites non mappés : {unmapped}")
else:
    print(" Tous les sites sont mappés")

print(f"\nFichiers par site :")
print(df_files.groupby("site")["filepath"].count().sort_values(ascending=False))

## 3. Calcul du volume moyen et de la carte tSD par site

Pour chaque site :
- **Volume moyen** : moyenne voxel par voxel de l'image fMRI moyenne de chaque sujet (`mean_img`)
- **Carte tSD** : écart-type voxel par voxel sur la série temporelle de chaque sujet, puis moyenne par site

On traite **un sujet à la fois** pour éviter de saturer la mémoire (chaque fichier ~100 Mo).

In [ ]:
def compute_site_maps(filepaths):
    """
    Calcule le volume moyen et la carte tSD par site.
    Utilise image.mean_img de nilearn pour le volume moyen.
    """
    mean_imgs = []
    sum_tsd  = None
    affine   = None
    n_valid  = 0

    for fp in filepaths:
        try:
            img  = nib.load(str(fp))
            # Volume moyen avec nilearn (plus propre)
            mean_img_subj = image.mean_img(img)
            mean_imgs.append(mean_img_subj)

            # Carte tSD manuellement
            data = img.get_fdata(dtype=np.float32)
            if data.ndim != 4:
                warnings.warn(f"{fp.name} : pas une image 4D, ignoré.", UserWarning)
                continue

            subj_tsd = data.std(axis=3)

            if sum_tsd is None:
                sum_tsd = subj_tsd.copy()
                affine  = img.affine
            else:
                if subj_tsd.shape != sum_tsd.shape:
                    ref_img  = nib.Nifti1Image(sum_tsd, affine)
                    new_tsd  = nib.Nifti1Image(subj_tsd, img.affine)
                    subj_tsd = image.resample_to_img(new_tsd, ref_img).get_fdata(dtype=np.float32)
                sum_tsd += subj_tsd

            n_valid += 1
            del data, subj_tsd

        except Exception as e:
            warnings.warn(f"{fp.name} : erreur ({e}), ignoré.", UserWarning)
            continue

    if n_valid == 0:
        return None, None

    # Utilise image.mean_img de nilearn pour la moyenne des volumes
    mean_map = image.mean_img(mean_imgs)
    tsd_map  = nib.Nifti1Image(sum_tsd / n_valid, affine)
    return mean_map, tsd_map

In [ ]:
# Calcul pour tous les sites
# ATTENTION Cette cellule peut prendre 15-30 minutes 

sites = sorted(df_files["site"].dropna().unique())
site_mean_maps = {}
site_tsd_maps  = {}

for site in sites:
    filepaths = df_files[df_files["site"] == site]["filepath"].tolist()
    print(f"Traitement site {site} ({len(filepaths)} sujets)...", end=" ", flush=True)

    mean_map, tsd_map = compute_site_maps(filepaths)

    if mean_map is not None:
        site_mean_maps[site] = mean_map
        site_tsd_maps[site]  = tsd_map
        print("ok")
    else:
        print("aucun fichier valide")

print(f"\nSites traités avec succès : {len(site_mean_maps)} / {len(sites)}")

In [ ]:
print(f"Sites dans site_mean_maps : {len(site_mean_maps)}")
print(list(site_mean_maps.keys()))

print(fmri_dir)
print(fmri_dir.exists())
print(len(list(fmri_dir.glob("*_func_preproc.nii.gz"))))

print(len(df_files))
print(df_files.head())
print(df_files["site"].value_counts())

print(list(fmri_dir.glob("*Leuven*"))[:3])

## 4. Visualisation des volumes moyens par site

On affiche une coupe axiale centrale pour chaque site. Les différences d'intensité ou de champ de vue entre sites sont directement visibles.

In [ ]:
sites_ok = sorted(site_mean_maps.keys())
n_sites  = len(sites_ok)
ncols    = 4
nrows    = int(np.ceil(n_sites / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for i, site in enumerate(sites_ok):
    plotting.plot_epi(
        site_mean_maps[site],
        axes=axes[i],
        title=site,
        display_mode="z",
        cut_coords=[0],
        colorbar=False,
        draw_cross=False
    )

# Masquer les axes vides
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Volume moyen fMRI par site", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("volume_moyen_par_site.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : volume_moyen_par_site.png")

## 5. Visualisation des cartes d'écart-type temporel (tSD) par site

La carte tSD révèle les zones avec une forte variabilité temporelle. Des valeurs élevées dans certaines régions (bords du cerveau, sinus, yeux) indiquent des artefacts dominants spécifiques au site.

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for i, site in enumerate(sites_ok):
    plotting.plot_stat_map(
        site_tsd_maps[site],
        axes=axes[i],
        title=site,
        display_mode="z",
        cut_coords=[0],
        colorbar=True,
        draw_cross=False,
        cmap="hot"
    )

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Carte d'écart-type temporel (tSD) par site", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("tSD_par_site.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : tSD_par_site.png")

## 6. Focus sur les sites problématiques vs performants

On compare côte à côte les sites qui généralisaient mal en LOSO (OHSU : 0.442, MAX_MUN : 0.499) avec un site performant (LEUVEN_1 : 0.750), pour voir si les différences visuelles correspondent aux différences de performance.

In [ ]:
# Sites sélectionnés pour la comparaison
# Basé sur les résultats LOSO de la tâche 1
sites_focus = {
    "LEUVEN_1":  "Bon (LOSO BA=0.750)",
    "PITT":      "Bon (LOSO BA=0.737)",
    "NYU":       "Moyen (LOSO BA=0.654)",
    "MAX_MUN":   "Problématique (LOSO BA=0.499)",
    "OHSU":      "Problématique (LOSO BA=0.442)",
}

fig, axes = plt.subplots(2, len(sites_focus), figsize=(len(sites_focus) * 4, 8))

for col, (site, label) in enumerate(sites_focus.items()):
    if site not in site_mean_maps:
        print(f"Attention : Site {site} non disponible")
        continue

    # Ligne 1 : volume moyen
    plotting.plot_epi(
        site_mean_maps[site],
        axes=axes[0, col],
        title=f"{site}\n{label}",
        display_mode="z",
        cut_coords=[0],
        colorbar=False,
        draw_cross=False
    )

    # Ligne 2 : carte tSD
    plotting.plot_stat_map(
        site_tsd_maps[site],
        axes=axes[1, col],
        display_mode="z",
        cut_coords=[0],
        colorbar=True,
        draw_cross=False,
        cmap="hot"
    )

axes[0, 0].set_ylabel("Volume moyen", fontsize=11)
axes[1, 0].set_ylabel("Carte tSD", fontsize=11)

fig.suptitle("Comparaison sites performants vs problématiques", fontsize=14)
plt.tight_layout()
plt.savefig("comparaison_sites_focus.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : comparaison_sites_focus.png")

In [ ]:
# Visualisation interactive : choisir un site
site_choisi = "NYU"  # changer selon le site voulu
view = plotting.view_img(site_mean_maps[site_choisi], 
                         title=f"Volume moyen — {site_choisi}")
view.save_as_html(f"volume_interactif_{site_choisi}.html")
view